In [ ]:
%load_ext autoreload
%autoreload 2

# Data Analysis with Neo

This notebook demonstrates a ChatGPT-like data analysis experience using Neo.
The LLM writes and executes Python code to analyze your dataset.

In [ ]:
import os
from neo.agentic import Neo, ModelTask, Instruction, ModelConfigs, OtherConfigs
from neo.tools.code_gen import code_execution_tool

## Configuration

In [ ]:
# Path to the dataset
DATASET_PATH = os.path.abspath("expense_log_nov2024.csv")
print(f"Dataset path: {DATASET_PATH}")

In [ ]:
DATA_ANALYSIS_PROMPT = f"""
You are an expert data analyst. Your task is to analyze datasets using Python code.

## Instructions
- Use the code_execution_tool to write and run Python code
- Always use language="python" when calling the tool
- The dataset is located at: {DATASET_PATH}
- Use pandas for data manipulation
- Always print results explicitly so they appear in the output
- For visualizations, save plots to files and describe what you observe
- Be thorough but concise in your analysis

## Analysis Guidelines
When asked to analyze a dataset, typically perform:
1. **Load & Inspect**: Load data, check shape, dtypes, first few rows
2. **Data Quality**: Check for missing values, duplicates, data types
3. **Summary Statistics**: Describe numerical and categorical columns
4. **Key Insights**: Identify patterns, trends, anomalies
5. **Recommendations**: Provide actionable insights based on findings

## Code Style
- Import libraries at the top of each script
- Use clear variable names
- Add brief comments for complex operations
- Print section headers to organize output
"""

## Register Code Execution Tool

Before using the tool with Neo, we need to register it in the ToolRegistry with a string code.

## Basic Analysis - Single Task

In [ ]:
# Register code_execution_tool with a string code so it can be used via OtherConfigs
from neo.agentic import ToolRegistry
tool_registry = ToolRegistry()
tool_registry.register_model_agnostic_tool(code="CodeGen", tool=code_execution_tool)

In [ ]:
instruction = Instruction(
    content=DATA_ANALYSIS_PROMPT,
    model_configs=ModelConfigs(
        model="gpt-5.2",
        reasoning={"effort": "low", "summary": "detailed"},
        include=["reasoning.encrypted_content"],
    ),
    other_configs=OtherConfigs(tools=["CodeGen"]),
)

task = ModelTask(
    user_input="Load the expense dataset and give me a comprehensive overview. What are the main spending categories? What's my total spending for the month?",
    instruction=instruction,
    id="Initial Analysis",
)

In [ ]:
neo = Neo(tasks=[task], max_tool_execution_rounds=10, model_registry_fuzzy_mode=True)
result = await neo.run_all(start_fresh=True)

In [ ]:
task.deliverable.display()

## Follow-up Questions

Continue the conversation with follow-up analysis questions.

In [ ]:
# Create a new task for follow-up, using the previous result as context
followup_task = ModelTask(
    user_input="What are my top 5 merchants by spending? Also, how much am I spending on recurring vs one-time expenses?",
    instruction=instruction,
    id="Follow-up Analysis"
)

neo2 = Neo(tasks=[followup_task], max_tool_execution_rounds=10)
result2 = await neo2.run_all(base_thread=result, start_fresh=True)

In [ ]:
result2.display()

## Two-Stage Analysis Pipeline with Reasoning Control

Chain multiple analysis tasks with different reasoning efforts. Stage 1 uses high reasoning to create a detailed analysis plan. Stage 2 uses low reasoning to efficiently execute that plan.

In [ ]:
# Stage 1: Planning with High Reasoning
planning_stage = ModelTask(
    user_input="Create a detailed, numbered analysis plan for the expense dataset. List specific steps for: 1) data loading and validation, 2) summary statistics to compute, 3) key questions to answer, 4) insights to extract. Be thorough and strategic.",
    instruction=Instruction(
        content=DATA_ANALYSIS_PROMPT,
        model_configs=ModelConfigs(
            model="gpt-5.2",
            reasoning={"effort": "high", "summary": "detailed"},
            include=["reasoning.encrypted_content"],
        ),
        other_configs=OtherConfigs(tools=["CodeGen"]),
    ),
    id="Planning Stage"
)

# Stage 2: Analysis with Low Reasoning
analysis_stage = ModelTask(
    user_input="Execute the analysis plan you just created. Run the code for each step and provide the results. Focus on execution efficiency.",
    instruction=Instruction(
        content=DATA_ANALYSIS_PROMPT,
        model_configs=ModelConfigs(
            model="gpt-5.2",
            reasoning={"effort": "low", "summary": "detailed"},
            include=["reasoning.encrypted_content"],
        ),
        other_configs=OtherConfigs(tools=["CodeGen"]),
    ),
    id="Analysis Stage"
)

# Chain the tasks
planning_stage.add_subsequent_task(analysis_stage)

In [ ]:
# Run the pipeline
workflow = Neo(tasks=[planning_stage], max_tool_execution_rounds=10)
workflow_result = await workflow.run_all(start_fresh=True)

In [ ]:
# Display task status
workflow.display_task_status()

In [ ]:
workflow_result.display()

## Debugging

Access individual task deliverables for debugging.

In [ ]:
# Get a specific task by ID
planning_task = workflow_result.get_task_by_id("Planning Stage")

# check unfinished deliverable
planning_task.unfinished_deliverable.display()